# SWE-Agent MoE Training Pipeline
## Full Multi-Stage Training on Google Colab

Architecture: ~38B total params, ~2.6B activated per token
Target: Qwen3.6-A3B-35B equivalence for SWE/agentic tasks

Phases:
1. Pretraining (SWE + math + language corpus)
2. Midtraining (reasoning data)
3. SFT (DataClaw datasets)
4. RL (SWE environments)

**Run all cells sequentially. Toggle the RUN_* booleans in each phase cell to enable.**

In [ ]:
# @title 1. Clone repo and install dependencies
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/BenItBuhner/swe-agent-moe.git"
PROJECT_ROOT = Path("/content/swe-agent-moe")

print("=" * 60)
print("SWE-Agent MoE ~38B Total / ~2.6B Activated")
print("Target: Qwen3.6-A3B-35B SWE/Agentic Performance")
print("=" * 60)

# Clone repo
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

py_files = len(list(PROJECT_ROOT.rglob("*.py")))
print(f"\nCloned {py_files} Python files from GitHub")

# Install dependencies
print("\nInstalling dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch>=2.4.0", "transformers>=4.44.0", "accelerate>=0.33.0",
    "datasets>=2.20.0", "wandb>=0.17.0", "sentencepiece>=0.2.0",
    "protobuf>=4.25.0", "einops>=0.7.0"])
print("Dependencies installed")

In [ ]:
# @title 2. Hardware Check
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Using GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device count: {torch.cuda.device_count()}")
    if gpu_mem >= 40:
        print("✓ High-memory GPU - full model training possible")
    elif gpu_mem >= 15:
        print("✓ Mid-range GPU - use CPU offloading or smaller batch")
    else:
        print("Use T4/V100 for quick tests, A100 for full training")
else:
    print("No GPU detected. Go to Runtime > Change runtime type > Select GPU/TPU.")

In [ ]:
# @title 3. Configuration
from configs.model_config import MoEModelConfig, TrainingConfig

model_config = MoEModelConfig()
train_config = TrainingConfig()

print("Model Configuration:")
print(f"  Layers: {model_config.num_hidden_layers}")
print(f"  Hidden: {model_config.hidden_size}")
print(f"  Heads: {model_config.num_attention_heads}")
print(f"  KV Heads: {model_config.num_kv_heads}")
print(f"  Experts: {model_config.num_experts} (top-{model_config.num_experts_per_tok})")
print(f"  Total params: {model_config.total_params:.2f}B")
print(f"  Activated params: {model_config.activated_params:.2f}B")
print(f"  Vocab: {model_config.vocab_size}")
print(f"  Max seq: {model_config.max_position_embeddings}")
print()
print("Training Configuration:")
print(f"  Pretrain steps: {train_config.pretrain_steps}")
print(f"  Midtrain steps: {train_config.midtrain_steps}")
print(f"  SFT steps: {train_config.sft_steps}")
print(f"  RL steps: {train_config.rl_steps}")

In [ ]:
# @title 4. Initialize Model
from model.architecture import MoEForCausalLM, MoEConfig
from transformers import AutoTokenizer

hf_config = MoEConfig(
    vocab_size=model_config.vocab_size,
    hidden_size=model_config.hidden_size,
    intermediate_size=model_config.intermediate_size,
    num_hidden_layers=model_config.num_hidden_layers,
    num_attention_heads=model_config.num_attention_heads,
    num_kv_heads=model_config.num_kv_heads,
    num_experts=model_config.num_experts,
    num_experts_per_tok=model_config.num_experts_per_tok,
    max_position_embeddings=model_config.max_position_embeddings,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", type(tokenizer).__name__)

In [ ]:
# @title 5. Phase 1: Pretraining
import sys, time, torch, wandb
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

RUN_PRETRAIN = False  # @param {type:"boolean"}
PRETRAIN_STEPS = 1000  # @param {type:"integer"}

if RUN_PRETRAIN:
    from data.pretraining_dataset import create_pretraining_dataloader

    model = MoEForCausalLM(hf_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        model = model.to(device)

    wandb.init(project="swe-agent-moe", config={"phase": "pretrain"})

    dataloader = create_pretraining_dataloader(
        tokenizer=tokenizer, batch_size=1, seq_length=4096,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, 100, PRETRAIN_STEPS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    os.makedirs("/content/checkpoints/pretrain", exist_ok=True)

    model.train()
    for step, batch in enumerate(dataloader):
        if step >= PRETRAIN_STEPS:
            break
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 10 == 0:
            print(f"Pretrain step {step}/{PRETRAIN_STEPS} | loss: {loss.item():.4f}")
            wandb.log({"pretrain_loss": loss.item(), "step": step})

        if step % 500 == 0 and step > 0:
            ckpt_path = f"/content/checkpoints/pretrain/checkpoint-{step}.pt"
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved checkpoint to {ckpt_path}")

    torch.save(model.state_dict(), "/content/checkpoints/pretrain/final.pt")
    wandb.finish()
    print("Phase 1: Pretraining complete!")
else:
    print("Skipping pretraining (set RUN_PRETRAIN = True to enable)")

In [ ]:
# @title 6. Phase 2: Midtraining (Reasoning)
RUN_MIDTRAIN = False  # @param {type:"boolean"}
MIDTRAIN_STEPS = 500  # @param {type:"integer"}

if RUN_MIDTRAIN:
    from data.midtraining_dataset import create_midtraining_dataloader

    model = MoEForCausalLM(hf_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt_path = "/content/checkpoints/pretrain/final.pt"
    if os.path.exists(ckpt_path):
        sd = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(sd, strict=False)
        model = model.to(device)
        print(f"Loaded pretrained checkpoint from {ckpt_path}")
    else:
        model = model.to(device)
        print("No pretrained checkpoint found, starting from scratch")

    wandb.init(project="swe-agent-moe", config={"phase": "midtrain"})

    dataloader = create_midtraining_dataloader(
        tokenizer=tokenizer, batch_size=1, seq_length=8192,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, 50, MIDTRAIN_STEPS)
    scaler = torch.cuda.amp.GradScaler()

    os.makedirs("/content/checkpoints/midtrain", exist_ok=True)

    model.train()
    for step, batch in enumerate(dataloader):
        if step >= MIDTRAIN_STEPS:
            break
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 10 == 0:
            print(f"Midtrain step {step}/{MIDTRAIN_STEPS} | loss: {loss.item():.4f}")
            wandb.log({"midtrain_loss": loss.item(), "step": step})

        if step % 200 == 0 and step > 0:
            torch.save(model.state_dict(), f"/content/checkpoints/midtrain/checkpoint-{step}.pt")

    torch.save(model.state_dict(), "/content/checkpoints/midtrain/final.pt")
    wandb.finish()
    print("Phase 2: Midtraining complete!")
else:
    print("Skipping midtraining (set RUN_MIDTRAIN = True to enable)")

In [ ]:
# @title 7. Phase 3: SFT (DataClaw)
RUN_SFT = False  # @param {type:"boolean"}
SFT_STEPS = 500  # @param {type:"integer"}

if RUN_SFT:
    from data.sft_dataset import create_sft_dataloader

    model = MoEForCausalLM(hf_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt_path = "/content/checkpoints/midtrain/final.pt"
    if os.path.exists(ckpt_path):
        sd = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(sd, strict=False)
        print(f"Loaded midtrain checkpoint from {ckpt_path}")
    else:
        model = model.to(device)
        print("No midtrain checkpoint, starting from scratch")

    wandb.init(project="swe-agent-moe", config={"phase": "sft"})

    dataloader = create_sft_dataloader(
        tokenizer=tokenizer, batch_size=1, seq_length=8192,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
    scheduler = get_cosine_schedule_with_warmup(optimizer, 50, SFT_STEPS)
    scaler = torch.cuda.amp.GradScaler()

    os.makedirs("/content/checkpoints/sft", exist_ok=True)

    model.train()
    for step, batch in enumerate(dataloader):
        if step >= SFT_STEPS:
            break
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 10 == 0:
            print(f"SFT step {step}/{SFT_STEPS} | loss: {loss.item():.4f}")
            wandb.log({"sft_loss": loss.item(), "step": step})

        if step % 200 == 0 and step > 0:
            torch.save(model.state_dict(), f"/content/checkpoints/sft/checkpoint-{step}.pt")

    torch.save(model.state_dict(), "/content/checkpoints/sft/final.pt")
    wandb.finish()
    print("Phase 3: SFT complete!")
else:
    print("Skipping SFT (set RUN_SFT = True to enable)")

In [ ]:
# @title 8. Phase 4: RL Training (SWE Environments)
RUN_RL = False  # @param {type:"boolean"}
RL_STEPS = 200  # @param {type:"integer"}

if RUN_RL:
    from rl.environments import SWEEnvironment, EnvResult
    from train.rl_train import train_rl_step

    model = MoEForCausalLM(hf_config)
    ref_model = MoEForCausalLM(hf_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt_path = "/content/checkpoints/sft/final.pt"
    if os.path.exists(ckpt_path):
        sd = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(sd, strict=False)
        ref_model.load_state_dict(sd, strict=False)
        model = model.to(device)
        ref_model = ref_model.to(device)
        print(f"Loaded SFT checkpoint from {ckpt_path}")

    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    env = SWEEnvironment()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=0.01)

    wandb.init(project="swe-agent-moe", config={"phase": "rl"})
    os.makedirs("/content/checkpoints/rl", exist_ok=True)

    model.train()
    for step in range(RL_STEPS):
        import random
        prompts = [
            "Write a Python function to find the longest common subsequence.",
            "Debug this code: def foo(x): return x / 0",
            "Refactor this function to use async/await.",
        ]
        batch = {"prompts": prompts}

        metrics = train_rl_step(
            model, ref_model, batch, env, optimizer, tokenizer,
            train_config, device,
        )

        if step % 10 == 0:
            print(f"RL step {step}/{RL_STEPS} | loss: {metrics['loss']:.4f} | reward: {metrics['reward']:.4f}")
            wandb.log({"rl_loss": metrics["loss"], "rl_reward": metrics["reward"], "step": step})

        if step % 100 == 0 and step > 0:
            torch.save(model.state_dict(), f"/content/checkpoints/rl/checkpoint-{step}.pt")

    torch.save(model.state_dict(), "/content/checkpoints/rl/final.pt")
    wandb.finish()
    print("Phase 4: RL training complete!")
else:
    print("Skipping RL (set RUN_RL = True to enable)")

In [ ]:
# @title 9. Evaluate Model on SWE Benchmarks
from scripts.benchmark_eval import evaluate_model

EVAL_CHECKPOINT = "/content/checkpoints/rl/final.pt"  # @param {type:"string"}

if os.path.exists(EVAL_CHECKPOINT):
    results = evaluate_model(EVAL_CHECKPOINT, max_new_tokens=1024, temperature=0.2)
    print(f"\nFinal average score: {results['average_score']:.3f}")
    print(f"Final average reward: {results['average_reward']:.3f}")
else:
    print(f"Checkpoint not found: {EVAL_CHECKPOINT}")
    print("Train a checkpoint first, or point to an existing one.")

In [ ]:
# @title 10. Save to HuggingFace Hub
SAVE_TO_HUB = False  # @param {type:"boolean"}
HF_REPO = "barnacle-agent/swe-agent-moe-v1"  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

if SAVE_TO_HUB and HF_TOKEN:
    from huggingface_hub import HfApi, HfFolder
    HfFolder.save_token(HF_TOKEN)

    model.push_to_hub(HF_REPO, use_auth_token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, use_auth_token=HF_TOKEN)
    print(f"Model saved to https://huggingface.co/{HF_REPO}")
else:
    print("Skipping HF Hub upload")